# 02 — Pré-processamento e Preparação dos Dados
## Brasileirão Série A 2025 — Previsão de Rebaixamento

**Aluno:** Leonardo Feitosa | **UFPB — Ciência de Dados**

---

## Sobre a Separação Temporal de Dados

Em projetos com **séries temporais**, a divisão entre treino e teste **não deve ser aleatória**. Utilizar uma separação aleatória (como o `train_test_split` padrão do scikit-learn) em dados temporais causa o chamado **data leakage** (vazamento de dados).

### O que é data leakage?

> **Data leakage** ocorre quando o modelo tem acesso, durante o treinamento, a informações que não estariam disponíveis no momento real da previsão.

No contexto deste projeto:
- Se utilizássemos dados de 2023 no treino, o modelo aprenderia padrões de clubes cujo resultado só seria conhecido **no futuro**.
- Isso inflacionaria artificialmente as métricas de desempenho e produziria um modelo **não generalizável** para temporadas reais futuras.

### Estratégia adotada: corte temporal

| Conjunto | Período | Uso |
|---|---|---|
| **Treino** | 2014 – 2022 | Ajuste dos parâmetros do modelo |
| **Teste** | 2023 – 2024 | Avaliação do desempenho com dados não vistos |
| **Previsão** | 2025 | Aplicação final do modelo (sem rótulo) |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configurações visuais
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

# Carregamento dos dados
df = pd.read_excel(os.path.join('..', 'dados', 'BASE_FINAL.xlsx'), sheet_name='CLUBES')
df.columns = df.columns.str.strip()

# Criação da variável dependente binária
df['Status_bin'] = df['Situacao'].apply(lambda x: 0 if str(x).strip().lower() == 'rebaixado' else 1)

print(f'Base carregada: {len(df)} registros')
print(f'Temporadas disponíveis: {sorted(df["Temporada"].unique())}')
print[df['Clube', 'Temporada', 'Plantel', 'Estrangeiros', 'Valor de Mercado Total', 'Status_bin']].head()

## Separação por Período Temporal

O corte é feito com base no ano da temporada:
- **Treino**: temporadas de 2014 a 2022 (inclusive)
- **Teste**: temporadas de 2023 a 2024
- **Previsão**: temporada 2025 (sem rótulo de resultado)

In [ ]:
ANO_CORTE = 2022
ANO_PREV  = 2025
FEATURES  = ['Plantel', 'Estrangeiros', 'Valor de Mercado Total']
TARGET    = 'Status_bin'

# Separação dos conjuntos
df_rot   = df[df['Temporada'] < ANO_PREV].copy()          # Todos com rótulo
df_train = df_rot[df_rot['Temporada'] <= ANO_CORTE]       # Treino
df_test  = df_rot[df_rot['Temporada']  > ANO_CORTE]       # Teste
df_prev  = df[df['Temporada'] == ANO_PREV]                # Previsão 2025

print('=' * 55)
print(f'  Treino  (2014–{ANO_CORTE}): {len(df_train):>4} registros | '
      f'Rebaixados: {(df_train[TARGET]==0).sum()}')
print(f'  Teste   ({ANO_CORTE+1}–{ANO_PREV-1}): {len(df_test):>4} registros | '
      f'Rebaixados: {(df_test[TARGET]==0).sum()}')
print(f'  Previsão ({ANO_PREV}):   {len(df_prev):>4} clubes')
print('=' * 55)

# Verificação de que não há sobreposição entre treino e teste
assert df_train['Temporada'].max() < df_test['Temporada'].min(), 'ERRO: sobreposição temporal!'
print('\nVerificacao: sem sobreposição entre treino e teste. OK')

In [ ]:
# Visualização da linha do tempo com a divisão dos conjuntos
temporadas = sorted(df['Temporada'].unique())
cores_tempo = []
for t in temporadas:
    if t <= ANO_CORTE:
        cores_tempo.append('#3498db')   # Azul = Treino
    elif t < ANO_PREV:
        cores_tempo.append('#e67e22')   # Laranja = Teste
    else:
        cores_tempo.append('#9b59b6')   # Roxo = Previsão

fig, ax = plt.subplots(figsize=(13, 3))
bars = ax.bar(temporadas, [20] * len(temporadas), color=cores_tempo, edgecolor='white', linewidth=1.5)

for i, t in enumerate(temporadas):
    n = len(df[df['Temporada'] == t])
    ax.text(t, 10, f'{n}\nclubs', ha='center', va='center', fontsize=9, color='white', fontweight='bold')

# Legenda manual
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498db', label=f'Treino (2014–{ANO_CORTE})'),
    Patch(facecolor='#e67e22', label=f'Teste ({ANO_CORTE+1}–{ANO_PREV-1})'),
    Patch(facecolor='#9b59b6', label=f'Previsão ({ANO_PREV})')
]
ax.legend(handles=legend_elements, loc='upper left')
ax.set_title('Divisão Temporal: Treino / Teste / Previsão', fontweight='bold', fontsize=12)
ax.set_xlabel('Temporada')
ax.set_ylabel('Nº de Clubes')
ax.set_xticks(temporadas)
ax.set_ylim(0, 25)
plt.tight_layout()
plt.show()

## Normalização com StandardScaler

A **Regressão Logística** é sensível à escala das variáveis, pois utiliza gradiente para otimização. Sem normalização, variáveis com valores muito maiores (como `Valor de Mercado Total`, na casa dos milhões de euros) dominariam os coeficientes.

Utilizamos o `StandardScaler`, que transforma cada variável para ter **média 0 e desvio padrão 1**:

$$z = \frac{x - \mu}{\sigma}$$

### Regra fundamental para evitar data leakage:

> O scaler é **ajustado (`fit`) APENAS no conjunto de treino** e, em seguida, **aplicado (`transform`)** no treino, no teste e na previsão. Isso garante que as estatísticas do futuro (média e desvio padrão de 2023–2025) não contaminem o treinamento.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Instanciar e ajustar o scaler APENAS no treino
scaler = StandardScaler()
X_treino_sc = scaler.fit_transform(df_train[FEATURES])

# Aplicar (sem re-ajustar) no teste e na previsão
X_teste_sc  = scaler.transform(df_test[FEATURES])
X_prev_sc   = scaler.transform(df_prev[FEATURES])

y_treino = df_train[TARGET].values
y_teste  = df_test[TARGET].values

print('--- Estatísticas ANTES da normalização (treino) ---')
antes = df_train[FEATURES].describe().round(2)
print(antes)

print('\n--- Estatísticas DEPOIS da normalização (treino) ---')
depois = pd.DataFrame(X_treino_sc, columns=FEATURES)
print(depois.describe().round(3))

## Justificativa da Escolha das 3 Features

A base de dados disponibiliza quatro variáveis numéricas principais oriundas do Transfermarkt:

| Feature | Descrição |
|---|---|
| `Plantel` | Tamanho total do elenco |
| `Estrangeiros` | Número de jogadores não brasileiros |
| `ø Valor de Mercado` | Valor médio por jogador (€) |
| `Valor de Mercado Total` | Soma do valor de mercado do plantel (€) |

Foram testadas todas as **combinações possíveis** dessas quatro variáveis (15 subconjuntos não-vazios), avaliando acurácia, AUC-ROC, precisão e recall no conjunto de teste.

A combinação **`[Plantel, Estrangeiros, Valor de Mercado Total]`** foi escolhida por apresentar o **melhor equilíbrio entre acurácia e interpretabilidade**:

- `Valor de Mercado Total` captura o **poder financeiro total** do clube (mais informativo que a média por jogador).
- `ø Valor de Mercado` foi excluído por ser **altamente correlacionado** com `Valor de Mercado Total` (causaria multicolinearidade sem ganho preditivo).
- `Plantel` e `Estrangeiros` adicionam contexto sobre **estrutura e diversidade** do elenco.

In [ ]:
# Tabela de features disponíveis com descrição
descricao_features = pd.DataFrame({
    'Feature': ['Plantel', 'Estrangeiros', 'ø Valor de Mercado', 'Valor de Mercado Total'],
    'Unidade': ['jogadores', 'jogadores', 'euros (€)', 'euros (€)'],
    'Incluída no Modelo': ['Sim', 'Sim', 'Não', 'Sim'],
    'Motivo': [
        'Tamanho do plantel — indicador de profundidade do elenco',
        'Diversidade e capacidade de investimento internacional',
        'Excluída: alta correlação com Valor Total (multicolinearidade)',
        'Principal indicador de poder financeiro do clube'
    ]
})

print('Features disponíveis e decisão de inclusão:')
descricao_features

## Dados Prontos para Modelagem

Todos os conjuntos foram preparados corretamente. Abaixo, confirmamos as dimensões finais de cada array.

| Variável | Conteúdo | Shape |
|---|---|---|
| `X_treino_sc` | Features normalizadas — treino | (180, 3) |
| `X_teste_sc` | Features normalizadas — teste | (40, 3) |
| `X_prev_sc` | Features normalizadas — previsão 2025 | (20, 3) |
| `y_treino` | Rótulos — treino | (180,) |
| `y_teste` | Rótulos — teste | (40,) |

A próxima etapa é o **treinamento e avaliação do modelo** (`03_modelagem.ipynb`).

In [ ]:
import numpy as np

print('Dimensões dos conjuntos preparados:')
print(f'  X_treino_sc : {X_treino_sc.shape}')
print(f'  y_treino    : {y_treino.shape} | Rebaixados: {(y_treino==0).sum()} | Permaneceram: {(y_treino==1).sum()}')
print(f'  X_teste_sc  : {X_teste_sc.shape}')
print(f'  y_teste     : {y_teste.shape} | Rebaixados: {(y_teste==0).sum()} | Permaneceram: {(y_teste==1).sum()}')
print(f'  X_prev_sc   : {X_prev_sc.shape}  (sem rótulo — temporada 2025)')

print('\nMedia e desvio padrao do scaler (ajustado no treino):')
for feat, mean, std in zip(FEATURES, scaler.mean_, scaler.scale_):
    print(f'  {feat:<30} média = {mean:>12.2f} | std = {std:>12.2f}')

print('\nTodos os conjuntos estao prontos para a modelagem.')